## Planning:
- Load data
- Maximum Parameter with Optuna
- Blending with Nelder Mead
- Output

In [2]:
import numpy as np
import pandas as pd
import optuna
from sklearn.metrics import roc_auc_score

# import numpy as np
from scipy.optimize import minimize
# from sklearn.metrics import roc_auc_score

# import OS
import os

In [11]:
from pathlib import Path

In [12]:
current_dir = Path.cwd()

In [15]:
competition_root = current_dir.parents[1]

In [18]:
# competition_root

In [19]:
competition_path = competition_root/"Predicting Credit Card Fraud"

In [20]:
train = pd.read_csv(competition_path/'data/train.csv')
train_data = train[["id","is_fraud"]]

test = pd.read_csv(competition_path/'data/test.csv')
test_data = test[["id"]]

submit = pd.read_csv(competition_path/'data/sample_submission.csv')

In [22]:
folder_oof = competition_path/"oof"
folder_sub = competition_path/"sub"

In [23]:
content_oof = os.listdir(folder_oof)
content_sub = os.listdir(folder_sub)

In [27]:
content_oof

['oof_catboost_42.csv',
 'oof_catboost_67.csv',
 'oof_catboost_bool_5_42.csv',
 'oof_catboost_cat_5_42.csv',
 'oof_catboost_num_5_67.csv',
 'oof_lgb_3.csv',
 'oof_lgb_te_106.csv',
 'oof_lgb_te_1508.csv',
 'oof_lgb_te_2312.csv',
 'oof_lgb_te_42.csv',
 'oof_lgb_te_67.csv',
 'oof_logit_bool_5_42.csv',
 'oof_logit_cat_5_42.csv',
 'oof_logit_num_5_42.csv',
 'oof_log_5cv_1.csv',
 'oof_mlp_42.csv',
 'oof_mlp_bool_42.csv',
 'oof_mlp_cat_42.csv',
 'oof_mlp_num_42.csv',
 'oof_xgb_2.csv',
 'oof_xgb_bool_5_42.csv',
 'oof_xgb_cat_5_42.csv',
 'oof_xgb_num_5_42.csv',
 'oof_xgb_te_106.csv',
 'oof_xgb_te_1508.csv',
 'oof_xgb_te_2312.csv',
 'oof_xgb_te_42.csv',
 'oof_xgb_te_67.csv',
 'oof_xgb_te_digitv2_2312.csv',
 'oof_xgb_te_digit_2312.csv',
 'oof_xgb_te_digit_42.csv',
 'oof_xgb_te_fe_106.csv',
 'oof_xgb_te_fe_1508.csv',
 'oof_xgb_te_fe_2312.csv',
 'oof_xgb_te_fe_42.csv',
 'oof_xgb_te_fe_67.csv']

In [ ]:
for oof in content_oof:
    path_oof = folder_oof/oof
    oof_data = pd.read_csv(path_oof)
    train_data = pd.concat([train_data,oof_data.iloc[:,[1]]],axis=1)

In [6]:
for sub in content_sub:
    path_sub = folder_sub+"/"+sub
    sub_data = pd.read_csv(path_sub)
    test_data = pd.concat([test_data,sub_data.iloc[:,[1]]],axis=1)

## Data Process for Modeling

In [7]:
X = train_data.drop(columns=["id","is_fraud"])
y = train_data["is_fraud"]
X_test = test_data.drop(columns=["id"])
X_test.columns = X.columns

In [8]:
'''
X = X.drop(columns=["log_5cv"])
X_test = X_test.drop(columns=["log_5cv"])
'''

'\nX = X.drop(columns=["log_5cv"])\nX_test = X_test.drop(columns=["log_5cv"])\n'

In [9]:
oof_matrix = X.values 
y_true_array = y.values
num_models = oof_matrix.shape[1]

In [10]:
num_models

33

In [14]:
# 2. Define the objective function with a penalty constraint
def objective(weights, predictions, y_true):
    # Standardize weights so they sum to 1 for the actual blend calculation
    # This helps Nelder-Mead evaluate valid combinations even during search
    sum_w = np.sum(weights)
    if sum_w == 0:
        normalized_weights = weights
    else:
        normalized_weights = weights / sum_w
    
    # Calculate the blended prediction
    blend = np.tensordot(normalized_weights, predictions, axes=((0), (1)))
    
    # Calculate our metric (Nelder-Mead MINIMIZES, so we do 1 - AUC)
    score = 1.0 - roc_auc_score(y_true, blend)
    
    # Penalty: If weights go negative, add a heavy penalty to steer the optimizer away
    penalty = 0
    if np.any(weights < 0):
        penalty += 10.0 * np.abs(np.min(weights))

    l1_lambda = 0.001
    penalty += l1_lambda * np.sum(np.abs(normalized_weights))
        
    return score + penalty

In [15]:
# 3. Setup and Run the Optimizer
# Start with equal weights for all 3 models
init_weights = np.array([1/num_models]*num_models)

In [16]:
# Run Nelder-Mead
res = minimize(
    objective, 
    init_weights, 
    args=(X, y), 
    method='Nelder-Mead',
    options={'maxiter': 2000, 'xatol': 1e-4, 'fatol': 1e-4}
)

In [17]:
# 4. Process and Normalize the Final Best Weights
best_raw_weights = res.x
best_weights = best_raw_weights / np.sum(best_raw_weights)

In [18]:
best_weights

array([1.40267025e-02, 1.35207477e-02, 6.17912262e-04, 6.10164706e-03,
       2.56930878e-02, 4.65933778e-02, 3.41302319e-02, 3.65167460e-02,
       1.76024296e-02, 3.62762721e-02, 1.81324507e-02, 8.27907070e-03,
       6.12421496e-05, 1.42976171e-02, 2.60644005e-02, 3.06134253e-02,
       3.60129705e-02, 9.93897369e-05, 1.52032807e-02, 7.33336734e-03,
       6.10456290e-02, 5.79544066e-02, 2.96429620e-02, 4.31039842e-02,
       5.64606394e-02, 3.70811927e-02, 4.48534903e-02, 5.78623538e-02,
       2.20857403e-02, 3.75295612e-02, 4.50653300e-02, 5.98314488e-02,
       6.03068921e-02])

In [19]:
# 5. Print Results
print("Optimization Successful:", res.success)
print(f"Optimized Blending Weights: {np.round(best_weights, 4)}")
print(f"Best ROC-AUC Score: {1.0 - res.fun:.4f}")

Optimization Successful: False
Optimized Blending Weights: [0.014  0.0135 0.0006 0.0061 0.0257 0.0466 0.0341 0.0365 0.0176 0.0363
 0.0181 0.0083 0.0001 0.0143 0.0261 0.0306 0.036  0.0001 0.0152 0.0073
 0.061  0.058  0.0296 0.0431 0.0565 0.0371 0.0449 0.0579 0.0221 0.0375
 0.0451 0.0598 0.0603]
Best ROC-AUC Score: 0.9461


In [20]:
# Compute the final blended predictions
final_blend = np.dot(X_test.values, best_weights)

In [21]:
submit["is_fraud"] = final_blend

In [22]:
submit.to_csv("Blending Nelder Mead v3.csv",index=False)

In [23]:
# Drop high correlated variable

In [24]:
# 1. Calculate the correlation matrix (if not already done)
corr_matrix = X.corr().abs()

# 2. Select the upper triangle of the correlation matrix
upper_tri = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

In [25]:
# 3. Find features with correlation greater than the threshold (e.g., 0.95)
threshold = 0.97
to_drop = [column for column in upper_tri.columns if any(upper_tri[column] > threshold)]

In [26]:
# 4. Drop the features from your training and test sets
X_reduced = X.drop(columns=to_drop)
X_test_reduced = X_test.drop(columns=to_drop)

In [28]:
oof_matrix = X_reduced.values 
y_true_array = y.values
num_models = oof_matrix.shape[1]

In [29]:
# 3. Setup and Run the Optimizer
# Start with equal weights for all 3 models
init_weights = np.array([1/num_models]*num_models)

In [30]:
# Run Nelder-Mead
res = minimize(
    objective, 
    init_weights, 
    args=(X_reduced, y), 
    method='Nelder-Mead',
    options={'maxiter': 2000, 'xatol': 1e-4, 'fatol': 1e-4}
)

In [31]:
# 4. Process and Normalize the Final Best Weights
best_raw_weights = res.x
best_weights = best_raw_weights / np.sum(best_raw_weights)

In [32]:
# 5. Print Results
print("Optimization Successful:", res.success)
print(f"Optimized Blending Weights: {np.round(best_weights, 4)}")
print(f"Best ROC-AUC Score: {1.0 - res.fun:.4f}")

Optimization Successful: True
Optimized Blending Weights: [6.448e-01 3.000e-04 0.000e+00 1.104e-01 6.190e-02 0.000e+00 1.826e-01]
Best ROC-AUC Score: 0.9413


In [33]:
# Compute the final blended predictions
final_blend = np.dot(X_test_reduced.values, best_weights)

In [34]:
submit["is_fraud"] = final_blend
submit.to_csv("Blending Nelder Mead v5.csv",index=False)